# 🍼 Gemma 4 E4B — Baby Cry Detection Fine-Tuning
**Fully automated. No human interruption required.**

This notebook:
1. Installs all dependencies
2. Downloads the Kaggle infant-cry dataset automatically
3. Converts all audio to WAV, preprocesses and validates every file
4. Builds train/val JSONL datasets
5. Loads Gemma 4 E4B in 4-bit with LoRA via Unsloth
6. Fine-tunes with SFTTrainer
7. Runs inference and saves the model

> **Runtime:** GPU (T4 or better). Run → Runtime → Change runtime type → T4 GPU.


## 1 — Install Dependencies

In [ ]:
%%capture
import os, re, sys

# ── Unsloth + training stack ──────────────────────────────────────────────────
if 'COLAB_' not in ''.join(os.environ.keys()):
    # Non-Colab (Kaggle, local, etc.)
    os.system('pip install unsloth')
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {
        '2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'
    }.get(v, '0.0.34')
    os.system('pip install sentencepiece protobuf "datasets==2.18.0" hf_transfer')
    os.system(
        f'pip install --no-deps unsloth_zoo bitsandbytes accelerate '
        f'{xformers} peft trl triton unsloth'
    )
    os.system('pip install --no-deps --upgrade "torchao>=0.16.0"')

# FIX BUG-01: single authoritative huggingface_hub pin (>=1.5.0 for Unsloth).
# Never downgrade this again further down the notebook.
os.system('pip install "huggingface_hub==1.5.0" --quiet')

# Pinned transformers + matching tokenizers (--no-deps: unsloth manages the rest)
os.system('pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"')

# Audio / data processing
os.system('apt-get install -y ffmpeg -qq')
os.system('pip install soundfile librosa kagglehub --quiet')
os.system('pip install torchcodec --quiet')  # FIX BUG-02: explicit codec install

import torch
torch._dynamo.config.recompile_limit = 64
print('✅ All dependencies installed.')


In [ ]:
%%capture
import os
os.system('pip install --no-deps --upgrade timm')  # Required for Gemma 4 vision/audio
print('✅ timm installed.')


## 2 — Global Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GLOBAL CONFIGURATION  — edit here if needed, nowhere else
# ══════════════════════════════════════════════════════════════════════════════

CFG = dict(
    # ── Dataset ───────────────────────────────────────────────────────────────
    kaggle_dataset   = 'sanmithasadhish/infant-cry-dataset',
    raw_csv          = None,          # auto-resolved after download
    converted_dir    = '/content/converted_audio',
    jsonl_dir        = '/content/data_jsonl',
    quarantine_dir   = '/content/quarantine',

    # ── Audio preprocessing ───────────────────────────────────────────────────
    target_sr        = 16000,         # Hz
    target_duration  = 4.0,           # seconds — pad/trim to this length
    silence_db       = 20,            # top_db for librosa silence trim
    min_duration     = 0.3,           # seconds — files shorter are quarantined

    # ── Labels ────────────────────────────────────────────────────────────────
    label_map        = {'yes': 'baby_cry', 'no': 'not_baby_cry'},
    instruction      = 'Is this audio a baby crying? Reply only: baby_cry or not_baby_cry.',

    # ── Train / val split ─────────────────────────────────────────────────────
    val_split        = 0.10,
    seed             = 42,

    # ── Model ─────────────────────────────────────────────────────────────────
    model_name       = 'unsloth/gemma-4-E4B-it',
    max_seq_length   = 8192,
    load_in_4bit     = True,

    # ── LoRA ──────────────────────────────────────────────────────────────────
    lora_r           = 16,
    lora_alpha       = 16,
    lora_dropout     = 0.0,
    target_modules   = ['q_proj','k_proj','v_proj','o_proj',
                        'gate_proj','up_proj','down_proj'],

    # ── SFT training ──────────────────────────────────────────────────────────
    per_device_batch = 2,             # safe for T4 14.5 GB
    grad_accum       = 4,             # effective batch = 8
    num_epochs       = 1,
    max_steps        = -1,            # -1 = use num_epochs
    learning_rate    = 5e-5,
    weight_decay     = 0.001,
    warmup_ratio     = 0.03,
    lr_scheduler     = 'cosine',
    logging_steps    = 5,
    save_steps       = 50,
    output_dir       = 'outputs',

    # ── Save / push ───────────────────────────────────────────────────────────
    lora_save_path   = 'gemma4_babycry_lora',
    push_to_hub      = False,         # set True + fill hf_repo to push
    hf_repo          = 'YOUR_HF_USER/gemma4-babycry-lora',
    hf_token         = '',            # leave empty if push_to_hub=False
)

import os, pathlib
for d in [CFG['converted_dir'], CFG['jsonl_dir'], CFG['quarantine_dir']]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print('✅ Configuration set.')
for k, v in CFG.items():
    print(f'   {k:25s} = {v}')


✅ Configuration set.
   kaggle_dataset            = sanmithasadhish/infant-cry-dataset
   raw_csv                   = None
   converted_dir             = /content/converted_audio
   jsonl_dir                 = /content/data_jsonl
   quarantine_dir            = /content/quarantine
   target_sr                 = 16000
   target_duration           = 4.0
   silence_db                = 20
   min_duration              = 0.3
   label_map                 = {'yes': 'baby_cry', 'no': 'not_baby_cry'}
   instruction               = Is this audio a baby crying? Reply only: baby_cry or not_baby_cry.
   val_split                 = 0.1
   seed                      = 42
   model_name                = unsloth/gemma-4-E4B-it
   max_seq_length            = 8192
   load_in_4bit              = True
   lora_r                    = 16
   lora_alpha                = 16
   lora_dropout              = 0.0
   target_modules            = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## 3 — Download Dataset (Kaggle, fully automatic)

In [ ]:
import kagglehub, pathlib, os

print(f'⬇  Downloading: {CFG["kaggle_dataset"]} ...')
dataset_root = kagglehub.dataset_download(CFG['kaggle_dataset'])
dataset_root = pathlib.Path(dataset_root)
print(f'✅ Dataset root: {dataset_root}')

# Auto-locate the CSV (handles different internal layouts)
csv_candidates = list(dataset_root.rglob('data.csv'))
assert csv_candidates, f'data.csv not found under {dataset_root}'
CFG['raw_csv'] = str(csv_candidates[0])
print(f'✅ CSV found: {CFG["raw_csv"]}')

# Store the root of the audio files
CFG['audio_root'] = str(csv_candidates[0].parent)
print(f'✅ Audio root: {CFG["audio_root"]}')


⬇  Downloading: sanmithasadhish/infant-cry-dataset ...


100%|██████████| 169M/169M [00:02<00:00, 79.1MB/s]

Extracting files...


✅ Dataset root: /root/.cache/kagglehub/datasets/sanmithasadhish/infant-cry-dataset/versions/1
✅ CSV found: /root/.cache/kagglehub/datasets/sanmithasadhish/infant-cry-dataset/versions/1/Dataset/data.csv
✅ Audio root: /root/.cache/kagglehub/datasets/sanmithasadhish/infant-cry-dataset/versions/1/Dataset


## 4 — Audio Conversion, Validation & Preprocessing

In [ ]:
# ── Audio utility functions ───────────────────────────────────────────────────
import subprocess, shutil, hashlib, json
import numpy as np
import soundfile as sf
import librosa
import pathlib
import torch
from transformers import TextStreamer  # used in run_inference streaming (optional)

QUARANTINE_LOG = []   # tracks quarantined files with reason
PROCESSED_LOG  = []   # tracks successfully processed files


def ffmpeg_convert(src: str, dst: str) -> bool:
    """Convert any audio format → WAV 16-bit PCM via ffmpeg. Returns True on success."""
    pathlib.Path(dst).parent.mkdir(parents=True, exist_ok=True)
    if pathlib.Path(dst).exists():
        return True
    try:
        subprocess.run(
            ['ffmpeg', '-y', '-i', src,
             '-ar', str(CFG['target_sr']),
             '-ac', '1',
             '-sample_fmt', 's16',
             '-loglevel', 'error', dst],
            check=True, capture_output=True
        )
        return True
    except subprocess.CalledProcessError:
        return False


def validate_and_preprocess(wav_path: str) -> tuple:
    """
    Load audio directly with soundfile (avoids AudioDecoder.array bug).
    Returns (numpy_array float32, sample_rate) or raises ValueError with reason.

    FIX BUG-04: silence check now happens AFTER librosa.effects.trim, not before.
    """
    if not pathlib.Path(wav_path).exists():
        raise ValueError('file_not_found')

    try:
        audio, sr = sf.read(wav_path, dtype='float32')
    except Exception as exc:
        raise ValueError(f'unreadable: {exc}')

    if audio.size == 0:
        raise ValueError('zero_length')

    # Stereo → mono
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Resample if needed
    if sr != CFG['target_sr']:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=CFG['target_sr'])
        sr = CFG['target_sr']

    # Clip detection — normalise if > 1% of samples saturate
    clip_fraction = np.mean(np.abs(audio) >= 0.999)
    if clip_fraction > 0.01:
        audio = audio / (np.abs(audio).max() + 1e-8)

    # Trim silence (do this BEFORE energy check — FIX BUG-04)
    audio, _ = librosa.effects.trim(audio, top_db=CFG['silence_db'])

    # Minimum duration check after trimming
    if len(audio) / sr < CFG['min_duration']:
        raise ValueError('too_short_after_trim')

    # Silent-audio check — AFTER trim so legitimate audio is not mis-quarantined
    if np.abs(audio).max() < 1e-5:
        raise ValueError('silent_audio')

    # Normalise amplitude to [-1, 1]
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak

    # Pad or truncate to target duration
    target_len = int(sr * CFG['target_duration'])
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    # NaN / Inf check
    if not np.isfinite(audio).all():
        raise ValueError('nan_or_inf_values')

    return audio.astype(np.float32), sr


def file_md5(path: str) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()


# ── Inference helpers (moved here from Baseline-Inference cell) ───────────────
# BUG-03 FIX: defined early so cells 7, 10, 11, 13 can all call them.

def load_audio_array(wav_path: str) -> np.ndarray:
    """
    Load a WAV → float32 numpy array, resampled to target_sr and normalised.
    Uses soundfile directly (avoids AudioDecoder.array AttributeError).
    """
    audio, sr = sf.read(wav_path, dtype='float32')
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != CFG['target_sr']:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=CFG['target_sr'])
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak
    return audio.astype(np.float32)


def run_inference(audio_array: np.ndarray, max_new_tokens: int = 32) -> str:
    """
    Run a single inference pass with the globally loaded model + processor.
    Audio is passed as a decoded numpy array — avoids AudioDecoder issues.
    """
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'audio', 'audio': audio_array},
                {'type': 'text',  'text':  CFG['instruction']},
            ]
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize              = True,
        return_dict           = True,
        return_tensors        = 'pt',
    ).to('cuda')

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
        )
    input_len  = inputs['input_ids'].shape[1]
    new_tokens = output_ids[0][input_len:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()


print('✅ Audio utility functions, load_audio_array, and run_inference defined.')


✅ Audio utility functions, load_audio_array, and run_inference defined.


In [ ]:
import pandas as pd, pathlib, shutil, os, collections
from tqdm.auto import tqdm

print(f'📂 Reading CSV: {CFG["raw_csv"]}')
df = pd.read_csv(CFG['raw_csv'])
print(f'   Total rows: {len(df)}')
print(f'   Columns   : {list(df.columns)}')
print(f'   Label dist:\n{df["is_cry"].value_counts().to_string()}')

records_ok  = []
records_bad = []
seen_md5    = {}

for _, row in tqdm(df.iterrows(), total=len(df), desc='Converting & validating'):
    raw_filename = row['filename'].strip()
    is_cry_raw   = str(row['is_cry']).strip().lower()

    # Resolve source path (CSV stores 'cry_data/...' but files are under dataset root)
    rel_part = raw_filename.replace('cry_data/', '', 1)
    src_path = str(pathlib.Path(CFG['audio_root']) / rel_part)

    stem     = pathlib.Path(rel_part).stem
    subdir   = pathlib.Path(rel_part).parent
    dst_path = str(pathlib.Path(CFG['converted_dir']) / subdir / (stem + '.wav'))

    if not pathlib.Path(src_path).exists():
        records_bad.append({'file': raw_filename, 'reason': 'source_not_found'})
        continue

    ok = ffmpeg_convert(src_path, dst_path)
    if not ok:
        records_bad.append({'file': raw_filename, 'reason': 'ffmpeg_conversion_failed'})
        shutil.copy(src_path, pathlib.Path(CFG['quarantine_dir']) / pathlib.Path(raw_filename).name)
        continue

    try:
        _, _ = validate_and_preprocess(dst_path)
    except ValueError as e:
        reason = str(e)
        records_bad.append({'file': raw_filename, 'reason': reason})
        shutil.copy(dst_path, pathlib.Path(CFG['quarantine_dir']) / pathlib.Path(dst_path).name)
        continue

    md5 = file_md5(dst_path)
    if md5 in seen_md5:
        records_bad.append({'file': raw_filename, 'reason': f'duplicate_of:{seen_md5[md5]}'}); continue
    seen_md5[md5] = raw_filename

    label = CFG['label_map'].get(is_cry_raw)
    if label is None:
        records_bad.append({'file': raw_filename, 'reason': f'unknown_label:{is_cry_raw}'}); continue

    records_ok.append({'wav_path': dst_path, 'label': label})
    PROCESSED_LOG.append({'file': raw_filename, 'label': label})  # FIX BUG-05

print(f'\n✅ Valid samples  : {len(records_ok)}')
print(f'⚠️  Quarantined    : {len(records_bad)}')
if records_bad:
    reasons = collections.Counter(r['reason'].split(':')[0] for r in records_bad)
    for reason, count in reasons.most_common():
        print(f'   {reason:35s}: {count}')

with open(f'{CFG["quarantine_dir"]}/quarantine_log.json', 'w') as f:
    json.dump(records_bad, f, indent=2)

label_counts = pd.Series([r['label'] for r in records_ok]).value_counts()
print(f'\nLabel distribution in valid set:')
print(label_counts.to_string())


📂 Reading CSV: /root/.cache/kagglehub/datasets/sanmithasadhish/infant-cry-dataset/versions/1/Dataset/data.csv
   Total rows: 889
   Columns   : ['is_cry', 'filename']
   Label dist:
is_cry
yes    565
no     324


Converting & validating:   0%|          | 0/889 [00:00<?, ?it/s]


✅ Valid samples  : 885
⚠️  Quarantined    : 4
   too_short_after_trim               : 4

Label distribution in valid set:
baby_cry        565
not_baby_cry    320


## 5 — Build Train / Val JSONL Datasets

In [ ]:
import json, random, pathlib

random.seed(CFG['seed'])
random.shuffle(records_ok)

val_size   = max(1, int(len(records_ok) * CFG['val_split']))
val_recs   = records_ok[:val_size]
train_recs = records_ok[val_size:]

print(f'Train samples: {len(train_recs)}')
print(f'Val   samples: {len(val_recs)}')  # FIX BUG-06: was len(val_size if False else val_recs)


def make_message(wav_path: str, label: str) -> dict:
    """Build a single chat-format record for SFTTrainer."""
    return {
        'messages': [
            {
                'role': 'user',
                'content': [
                    {'type': 'audio', 'audio': wav_path},
                    {'type': 'text',  'text':  CFG['instruction']},
                ]
            },
            {
                'role': 'assistant',
                'content': [{'type': 'text', 'text': label}]
            }
        ]
    }


def write_jsonl(records, path):
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(make_message(r['wav_path'], r['label'])) + '\n')
    print(f'  ✔  {len(records):>4} examples → {path}')


train_jsonl = f'{CFG["jsonl_dir"]}/train.jsonl'
val_jsonl   = f'{CFG["jsonl_dir"]}/val.jsonl'

write_jsonl(train_recs, train_jsonl)
write_jsonl(val_recs,   val_jsonl)

print('\n✅ JSONL files written.')


Train samples: 797
Val   samples: 88
  ✔   797 examples → /content/data_jsonl/train.jsonl
  ✔    88 examples → /content/data_jsonl/val.jsonl

✅ JSONL files written.


## 6 — Load Gemma 4 E4B (4-bit, LoRA)

In [ ]:
from unsloth import FastModel
import torch

print(f'Loading model: {CFG["model_name"]} ...')

model, processor = FastModel.from_pretrained(
    model_name     = CFG['model_name'],
    dtype          = None,              # auto-detect (fp16 on T4, bf16 on A100)
    max_seq_length = CFG['max_seq_length'],
    load_in_4bit   = CFG['load_in_4bit'],
    # FIX BUG-07: removed full_finetuning=False (not a valid parameter)
)

print('✅ Base model loaded.')
print(f'   dtype       : {next(model.parameters()).dtype}')
print(f'   device      : {next(model.parameters()).device}')


/tmp/ipykernel_3508/2350728367.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastModel


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers    = True,
    finetune_language_layers  = True,
    finetune_attention_modules= True,
    finetune_mlp_modules      = True,
    r                         = CFG['lora_r'],
    lora_alpha                = CFG['lora_alpha'],
    lora_dropout              = CFG['lora_dropout'],
    bias                      = 'none',
    random_state              = CFG['seed'],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA applied.')
print(f'   Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)')


## 7 — Baseline Inference (Pre-Training)

Demonstrates that audio loading and the full inference pipeline work **before** fine-tuning.


In [ ]:
# load_audio_array and run_inference are defined in Cell-09 (Audio Utilities).
# No extra installs or re-imports needed here.

print('── Baseline Inference (pre-training) ────────────────────────')
for label_target in ['baby_cry', 'not_baby_cry']:
    sample = next((r for r in val_recs if r['label'] == label_target), None)
    if sample is None:
        print(f'⚠  No {label_target} sample in val set.')
        continue
    audio_arr = load_audio_array(sample['wav_path'])
    pred      = run_inference(audio_arr)
    match     = '✅' if pred == label_target else '❌'
    print(
        f'{match} Expected: {label_target:15s}  |  '
        f'Predicted: {pred}  |  '
        f'File: {sample["wav_path"].split("/")[-1]}'
    )

print('\n✅ Baseline inference complete.')


## 8 — Prepare Dataset for SFTTrainer

In [ ]:
from datasets import load_dataset
import soundfile as sf
import librosa
import numpy as np

# Load raw JSONL
raw_dataset = load_dataset(
    'json',
    data_files={
        'train':      train_jsonl,
        'validation': val_jsonl,
    }
)
print('Raw dataset:', raw_dataset)


def format_intersection_data(examples: dict) -> dict:
    """
    Decode WAV paths in each example's messages → float32 numpy arrays.

    FIX BUG-11: does NOT call apply_chat_template here.
    UnslothVisionDataCollator will call the processor (tokenize=True) during
    collation, which is the correct place for audio feature extraction.

    Returns {'messages': [...]}.  Samples that fail audio loading are dropped.
    """
    out_messages = []

    for msg_list in examples['messages']:
        user_msg = msg_list[0]
        asst_msg = msg_list[1]
        wav_path = user_msg['content'][0]['audio']

        try:
            audio_arr, sr = sf.read(wav_path, dtype='float32')
            if audio_arr.ndim > 1:
                audio_arr = audio_arr.mean(axis=1)
            if sr != CFG['target_sr']:
                audio_arr = librosa.resample(audio_arr, orig_sr=sr, target_sr=CFG['target_sr'])
            peak = np.abs(audio_arr).max()
            if peak > 0:
                audio_arr = audio_arr / peak
            target_len = int(CFG['target_sr'] * CFG['target_duration'])
            if len(audio_arr) < target_len:
                audio_arr = np.pad(audio_arr, (0, target_len - len(audio_arr)))
            else:
                audio_arr = audio_arr[:target_len]

            out_messages.append([
                {
                    'role': 'user',
                    'content': [
                        # .tolist() → JSON-serialisable; collator converts to tensor
                        {'type': 'audio', 'audio': audio_arr.tolist()},
                        {'type': 'text',  'text':  user_msg['content'][1]['text']},
                    ]
                },
                {
                    'role': 'assistant',
                    'content': asst_msg['content'],
                }
            ])
        except Exception:
            continue   # silently drop bad samples

    if not out_messages:
        return {'messages': []}
    return {'messages': out_messages}


print('Applying format_intersection_data to train split...')
train_dataset = raw_dataset['train'].map(
    format_intersection_data,
    batched=True,
    batch_size=8,
    remove_columns=raw_dataset['train'].column_names,
)
print(f'✅ Train dataset ready: {len(train_dataset)} examples')

print('Applying format_intersection_data to validation split...')
val_dataset = raw_dataset['validation'].map(
    format_intersection_data,
    batched=True,
    batch_size=8,
    remove_columns=raw_dataset['validation'].column_names,
)
print(f'✅ Val dataset ready  : {len(val_dataset)} examples')


## 9 — Fine-Tune with SFTTrainer

Key settings:
- `per_device_train_batch_size=2` + `gradient_accumulation_steps=4` (safe for T4 14.5 GB)
- `num_train_epochs=3`
- `UnslothVisionDataCollator` handles audio token alignment
- `dataset_kwargs={'skip_prepare_dataset': True}` required for audio
- `dataset_text_field=''` required for audio


In [ ]:
import torch
gpu_stats        = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory       = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU = {gpu_stats.name}. Max memory = {max_memory} GB.')
print(f'{start_gpu_memory} GB of memory already reserved.')


In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

max_steps_val  = -1 if CFG['max_steps'] == -1 else CFG['max_steps']
num_epochs_val = CFG['num_epochs'] if max_steps_val == -1 else 1

trainer = SFTTrainer(
    model            = model,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    processing_class = processor.tokenizer,
    data_collator    = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        # ── Batch & gradient ──────────────────────────────────────────────────
        per_device_train_batch_size  = CFG['per_device_batch'],
        per_device_eval_batch_size   = CFG['per_device_batch'],
        gradient_accumulation_steps  = CFG['grad_accum'],

        # ── Epochs / steps ────────────────────────────────────────────────────
        num_train_epochs  = num_epochs_val,
        max_steps         = max_steps_val,

        # ── LR schedule ───────────────────────────────────────────────────────
        learning_rate     = CFG['learning_rate'],
        warmup_ratio      = CFG['warmup_ratio'],
        lr_scheduler_type = CFG['lr_scheduler'],
        weight_decay      = CFG['weight_decay'],

        # ── Optimiser ─────────────────────────────────────────────────────────
        optim             = 'adamw_8bit',
        fp16              = not torch.cuda.is_bf16_supported(),
        bf16              = torch.cuda.is_bf16_supported(),

        # ── Logging & saving ──────────────────────────────────────────────────
        logging_steps     = CFG['logging_steps'],
        save_strategy     = 'steps',
        save_steps        = CFG['save_steps'],
        eval_strategy     = 'steps',
        eval_steps        = CFG['save_steps'],
        load_best_model_at_end = True,
        metric_for_best_model  = 'eval_loss',
        output_dir        = CFG['output_dir'],
        report_to         = 'none',
        seed              = CFG['seed'],

        # ── Audio fine-tuning — required flags ────────────────────────────────
        dataset_text_field           = '',
        dataset_kwargs               = {'skip_prepare_dataset': True},
        max_length                   = CFG['max_seq_length'],
        remove_unused_columns        = False,
    )
)

print('✅ SFTTrainer configured.')


In [ ]:
print('🚀 Starting fine-tuning...')
trainer_stats = trainer.train()
print('\n✅ Training complete!')


In [ ]:
import torch

# FIX BUG-13: guard against Cell-21 being skipped
if 'start_gpu_memory' not in dir() or 'max_memory' not in dir():
    gpu_stats        = torch.cuda.get_device_properties(0)
    start_gpu_memory = 0.0
    max_memory       = round(gpu_stats.total_memory / 1024**3, 3)

used_memory    = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_for_lora  = round(used_memory - start_gpu_memory, 3)
used_pct       = round(used_memory / max_memory * 100, 2)
lora_pct       = round(used_for_lora / max_memory * 100, 2)
runtime_sec    = trainer_stats.metrics['train_runtime']

print(f'Training time          : {runtime_sec:.1f} s  ({runtime_sec/60:.2f} min)')
print(f'Peak GPU memory        : {used_memory} GB  ({used_pct}% of {max_memory} GB)')
print(f'Memory used for LoRA   : {used_for_lora} GB  ({lora_pct}% of {max_memory} GB)')
print(f'Train loss (final)     : {trainer_stats.metrics.get("train_loss", "N/A")}')


## 10 — Post-Training Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# Switch to inference mode (re-enable after trainer potentially put model in train mode)
FastModel.for_inference(model)

y_true, y_pred = [], []
errors = 0

for rec in val_recs:
    try:
        audio_arr = load_audio_array(rec['wav_path'])
        pred      = run_inference(audio_arr, max_new_tokens=16)
        pred_norm = pred.strip().lower()
        if 'baby_cry' in pred_norm and 'not' not in pred_norm:
            pred_norm = 'baby_cry'
        elif 'not' in pred_norm or pred_norm == 'not_baby_cry':
            pred_norm = 'not_baby_cry'
        y_true.append(rec['label'])
        y_pred.append(pred_norm)
    except Exception as e:
        errors += 1

print(f'Evaluated {len(y_true)} samples  |  {errors} errors skipped')
print('\n── Classification Report ─────────────────────────────────────')
print(classification_report(y_true, y_pred, digits=4))
print('── Confusion Matrix ──────────────────────────────────────────')
labels = ['baby_cry', 'not_baby_cry']
cm     = confusion_matrix(y_true, y_pred, labels=labels)
cm_df  = pd.DataFrame(cm,
                      index   = [f'True:{l}' for l in labels],
                      columns = [f'Pred:{l}' for l in labels])
print(cm_df.to_string())


## 11 — Inference Demo (Post Fine-Tuning)

In [ ]:
# json was imported in Cell-09; no re-import needed. FIX BUG-14.

print('── Post-Fine-Tuning Inference Demo ──────────────────────────')
for label_target in ['baby_cry', 'not_baby_cry']:
    sample = next((r for r in val_recs if r['label'] == label_target), None)
    if sample is None:
        continue
    audio_arr = load_audio_array(sample['wav_path'])
    pred      = run_inference(audio_arr, max_new_tokens=16)
    match     = '✅' if pred.strip() == label_target else '❌'
    result = {
        'file_name'    : sample['wav_path'].split('/')[-1],
        'prediction'   : pred.strip(),
        'ground_truth' : label_target,
        'match'        : match,
    }
    print(json.dumps(result, indent=2))


## 12 — Save Model (LoRA Adapters + Optional Push to Hub)

In [ ]:
import os

lora_path = CFG['lora_save_path']
model.save_pretrained(lora_path)
processor.save_pretrained(lora_path)
print(f'✅ LoRA adapters saved to: {lora_path}')

if CFG['push_to_hub'] and CFG['hf_token'] and CFG['hf_repo']:
    model.push_to_hub(CFG['hf_repo'], token=CFG['hf_token'])
    processor.push_to_hub(CFG['hf_repo'], token=CFG['hf_token'])
    print(f'✅ Pushed to Hub: {CFG["hf_repo"]}')
else:
    print('ℹ️  push_to_hub=False — skipping Hub upload.')


In [ ]:
SAVE_GGUF = True  # set True for llama.cpp / GGUF export

if SAVE_GGUF:
    model.save_pretrained_gguf('gemma4_babycry_gguf', processor, quantization_method='q8_0')
    print('✅ GGUF model saved to: gemma4_babycry_gguf')
else:
    print('ℹ️  SAVE_GGUF=False — skipping GGUF export.')


In [ ]:
SAVE_MERGED = False  # set True to save full merged float16 model

if SAVE_MERGED:
    model.save_pretrained_merged('gemma4_babycry_merged_f16', processor)
    print('✅ Merged float16 model saved to: gemma4_babycry_merged_f16')
else:
    print('ℹ️  SAVE_MERGED=False — skipping merged model export.')


## 13 — Reload & Verify LoRA (Optional Smoke Test)

In [ ]:
# !pip install llama-cpp-python


In [ ]:
import gc, torch

RELOAD_TEST = True   # set False to skip

if RELOAD_TEST:

    # ── 1. Kill trainer FIRST (it owns the model + optimizer states) ──────────
    print('🧹 Freeing training objects from VRAM ...')
    for _name in ['trainer', 'model', 'processor',
                  'train_dataset', 'val_dataset', 'inputs', 'out']:
        try:    exec(f'del {_name}', globals())
        except: pass
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    _free  = (torch.cuda.get_device_properties(0).total_memory
              - torch.cuda.memory_reserved()) / 1024**3
    _alloc = torch.cuda.memory_allocated() / 1024**3
    print(f'   Allocated: {_alloc:.2f} GB  |  Free: {_free:.2f} GB')

    # ── 2. Guard — skip gracefully if still not enough headroom ──────────────
    if _free < 9.5:
        print(
            '⚠️  Only {:.2f} GB free — not enough to reload a 4-bit Gemma-4-E4B (~10 GB).\n'
            '   ➜  The saved LoRA in "{}" is valid; smoke-test skipped.\n'
            '   ➜  To verify locally: download the LoRA and run Cell-14.'.format(
                _free, CFG['lora_save_path'])
        )
    else:
        # ── 3. Reload ─────────────────────────────────────────────────────────
        from unsloth import FastModel as FM
        print(f'\n⬇  Loading LoRA from: {CFG["lora_save_path"]} ...')
        model, processor = FM.from_pretrained(
            model_name     = CFG['lora_save_path'],
            max_seq_length = CFG['max_seq_length'],
            load_in_4bit   = CFG['load_in_4bit'],
        )
        FM.for_inference(model)
        print('✅ Reloaded.')

        # ── 4. Smoke-test ─────────────────────────────────────────────────────
        sample    = val_recs[0]
        audio_arr = load_audio_array(sample['wav_path'])
        inputs = processor.apply_chat_template(
            [{'role': 'user', 'content': [
                {'type': 'audio', 'audio': audio_arr},
                {'type': 'text',  'text':  CFG['instruction']},
            ]}],
            add_generation_prompt=True,
            tokenize=True, return_dict=True, return_tensors='pt',
        ).to('cuda')
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=16, do_sample=False)
        answer = processor.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        ).strip()
        match = '✅' if answer == sample['label'] else '❌'
        print(f'\n{match} Prediction  : {answer}')
        print(f'   Ground truth: {sample["label"]}')

## 14 — Export & Auto-Download to Your Browser

Three formats available — pick what suits your local setup:

| Format | Size (approx.) | Use case |
|--------|---------------|----------|
| **LoRA adapters** (safetensors) | ~150 MB | Load with Unsloth / PEFT on top of the base model |
| **Merged float16** (safetensors) | ~9 GB | Full standalone model; load anywhere |
| **GGUF q4_K_M** | ~2.5 GB | Run locally with llama.cpp / Ollama / LM Studio |

Run the cell below — it zips your chosen format and your browser
will start the download automatically (Colab `files.download`).


In [ ]:
import os, shutil, pathlib, gc, torch
from google.colab import files   # Colab-only — triggers automatic browser download

# ══════════════════════════════════════════════════════════════════════════════
#  CHOOSE what to export  (set exactly ONE to True)
# ══════════════════════════════════════════════════════════════════════════════
EXPORT_LORA         = True    # LoRA adapters only  (~150 MB zip)  ← recommended
EXPORT_MERGED_F16   = False   # merged float16       (~9 GB zip)
EXPORT_GGUF_Q4      = True   # GGUF q4_K_M          (~2.5 GB)

ZIP_NAME            = '/content/gemma4_babycry_export.zip'

# ── helpers ───────────────────────────────────────────────────────────────────
def _free_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def _zip_and_download(folder: str, zip_path: str):
    if pathlib.Path(zip_path).exists():
        os.remove(zip_path)
    print(f'📦 Zipping {folder} → {zip_path} ...')
    shutil.make_archive(zip_path.replace('.zip',''), 'zip', folder)
    size_mb = os.path.getsize(zip_path) / 1024**2
    print(f'   Size: {size_mb:.1f} MB')
    print('⬇  Starting browser download ...')
    files.download(zip_path)

# ─────────────────────────────────────────────────────────────────────────────
if EXPORT_LORA:
    # LoRA adapters were already saved by Cell-30; just zip them.
    lora_dir = CFG['lora_save_path']
    assert pathlib.Path(lora_dir).exists(), f'LoRA dir not found: {lora_dir}'
    _zip_and_download(lora_dir, ZIP_NAME)

elif EXPORT_MERGED_F16:
    # Free VRAM, merge weights, save, zip, download.
    merged_dir = '/content/gemma4_babycry_merged_f16'
    if not pathlib.Path(merged_dir).exists():
        print('🔀 Merging LoRA into base model (float16) — this takes ~2 min ...')
        _free_vram()
        model.save_pretrained_merged(merged_dir, processor,
                                      save_method='merged_16bit')
        print('✅ Merge complete.')
    _zip_and_download(merged_dir, ZIP_NAME)

elif EXPORT_GGUF_Q4:
    # Merge + quantise to GGUF q4_K_M, then download the single .gguf file.
    gguf_dir  = '/content/baby-gemma'
    if not pathlib.Path(gguf_dir).exists():
        print('🔀 Converting to GGUF q4_K_M — this takes ~3-5 min ...')
        _free_vram()
        model.save_pretrained_gguf(gguf_dir, processor,
                                    quantization_method='q4_k_m')
        print('✅ GGUF conversion complete.')
    # GGUF produces a single file; download it directly (no zip needed).
    gguf_files = list(pathlib.Path(gguf_dir).glob('*.gguf'))
    assert gguf_files, 'No .gguf file found after conversion!'
    gguf_file = str(gguf_files[0])
    size_mb   = os.path.getsize(gguf_file) / 1024**2
    print(f'   File: {gguf_file}  ({size_mb:.0f} MB)')
    print('⬇  Starting browser download ...')
    files.download(gguf_file)

else:
    print('⚠️  No export format selected. Set EXPORT_LORA / EXPORT_MERGED_F16 / EXPORT_GGUF_Q4 = True.')

print('\n✅ Export cell finished.')


In [ ]:
# Run this in Colab after training
model.save_pretrained_gguf(
    "gemma4_babycry_gguf",          # folder name
    processor,
    quantization_method = "q4_k_m" # good balance of size vs quality
)
print("✅ Saved.")

In [ ]:
from google.colab import files
import os

gguf_file = [
    f for f in os.listdir("gemma4_babycry_gguf") if f.endswith(".gguf")
][0]

full_path = f"gemma4_babycry_gguf/{gguf_file}"
print(f"Downloading: {full_path}  ({os.path.getsize(full_path)/1024**3:.1f} GB)")
files.download(full_path)

## 15 — Load the Downloaded Model on Your Local Machine

### A) LoRA adapters (requires base model)
```python
from unsloth import FastModel
model, processor = FastModel.from_pretrained(
    model_name   = 'unsloth/gemma-4-E4B-it',   # base model
    load_in_4bit = True,
)
# Then load LoRA on top:
from peft import PeftModel
model = PeftModel.from_pretrained(model, '/path/to/gemma4_babycry_lora')
```

### B) Merged float16 (standalone, no base model needed)
```python
from unsloth import FastModel
model, processor = FastModel.from_pretrained(
    model_name   = '/path/to/gemma4_babycry_merged_f16',
    load_in_4bit = True,   # re-quantise on load to save VRAM
)
```

### C) GGUF with llama.cpp / Ollama
```bash
# llama.cpp
./llama-cli -m gemma4_babycry-q4_k_m.gguf -p "Is this audio a baby crying?"

# Ollama (create a Modelfile first)
echo 'FROM ./gemma4_babycry-q4_k_m.gguf' > Modelfile
ollama create babycry -f Modelfile
ollama run babycry
```


## 16 — Run Summary

All steps completed without human intervention:

| Step | Description | Status |
|------|-------------|--------|
| 1 | Install dependencies | Auto |
| 2 | Global configuration | Auto |
| 3 | Download Kaggle dataset | Auto |
| 4 | Convert, validate, preprocess audio | Auto |
| 5 | Build train/val JSONL | Auto |
| 6 | Load Gemma 4 E4B + LoRA | Auto |
| 7 | Baseline inference (pre-training) | Auto |
| 8 | Format dataset for SFTTrainer | Auto |
| 9 | Fine-tune | Auto |
| 10 | Evaluate (precision/recall/F1) | Auto |
| 11 | Inference demo | Auto |
| 12 | Save LoRA adapters | Auto |
| 13 | Reload & smoke test | Auto |

---

### Bugs fixed in this version

| # | Cell | Original bug | Fix applied |
|---|------|-------------|-------------|
| 01 | Install | `huggingface_hub==1.5.0` installed in Cell-1 then force-downgraded to `0.20.0` in Cell-7, breaking Unsloth at runtime | Removed the force-reinstall entirely; single pin `1.5.0` throughout |
| 02 | Install | `torchcodec` not explicitly included in Colab deps list | Added explicit `pip install torchcodec` |
| 03 | Audio utilities | `load_audio_array` / `run_inference` only defined in Cell-7 (Baseline Inference) — cells 10, 11, 13 depend on them but run after Cell-7 may be absent | Moved both functions to Cell-09 (Audio Utilities) so they're always defined early |
| 04 | Audio utilities | `validate_and_preprocess`: silent-audio energy check ran *before* silence trim, causing valid files with heavy padding to be quarantined | Reordered: trim silence first, then check energy |
| 05 | Audio processing | `PROCESSED_LOG` declared but never populated | Added `PROCESSED_LOG.append(...)` on success to mirror `QUARANTINE_LOG` |
| 06 | Build JSONL | `print(f'Val samples: {len(val_size if False else val_recs)}')` — tautological dead-code guard; would `TypeError` if condition ever changed | Replaced with `len(val_recs)` |
| 07 | Load model | `full_finetuning=False` passed to `FastModel.from_pretrained` — not a valid kwarg → `TypeError` | Removed the parameter |
| 08 | Baseline inference | Entire force-reinstall block + redundant `import unsloth` / `from unsloth import FastModel` | Removed; model is already loaded from Cell-14 |
| 09 | Baseline inference | `TextStreamer` imported but never referenced | Import removed |
| 10 | Baseline inference | Defensive `if 'model' not in locals()` re-load guard was masking real errors | Removed; model is guaranteed by Cell-14/15 |
| 11 | Format dataset | `apply_chat_template(tokenize=False)` called inside `map()` — produces only a text template, no audio features; collator then received incomplete data | Removed that call; return only `{'messages': [...]}` so collator handles full tokenisation |
| 12 | Format dataset | Dead `text` output key returned alongside `messages`; collator ignored it but caused confusion | Removed |
| 13 | Memory stats | `start_gpu_memory` / `max_memory` referenced from Cell-21 — `NameError` if Cell-21 was skipped | Added safe fallback initialisation |
| 14 | Inference demo | `import json as _json` — redundant (json already imported Cell-09) and opaque alias | Removed; use `json` directly |
